# Bonus: the clock *is* the speed

A synchronous circuit does one step of work per clock tick. So latency is (cycles x clock
period): for a fixed design, a faster clock is literally a faster accelerator - until the
clock gets *so* fast that signals can't settle within one period (the **critical path**)
and the outputs go wrong. That boundary is **timing closure**: the tools compute a
worst-case slack and you are "supposed" to stay under the rated clock.

The honest caveat for this board: static timing is *worst-case*, so real silicon usually
runs well past its rating before failing. On this PYNQ-Z2 the accelerator kept producing
correct pixels up to ~150-200 MHz; above that the AXI bus started hanging before we saw a
clean per-pixel "cliff". So treat the latency-vs-clock curve below as the reliable lesson
(speed scales with clock) and the correctness edge as fuzzy and board-dependent.

> **Board only.** The rungs below need the PYNQ board (`pynq` + the bitstream). On a
> laptop `available_backends()` omits them and the cell prints a skip message - that is
> expected; run this one on the board.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

In [ ]:
# Bounded clock sweep over the naive HLS accelerator. Keep MAX conservative - a too-high
# fclk0 can wedge a DMA and need an overlay reload. Restores 100 MHz on the way out.
CLOCK_SWEEP_MHZ = [100, 110, 120, 130, 140, 150]
RAMP_KERNEL = 'edges'


def clock_ramp(clocks_mhz=CLOCK_SWEEP_MHZ, kernel_name=RAMP_KERNEL, n=96):
    from fpga_conv import get_backend
    if 'hls_naive' not in available_backends():
        print('clock ramp is board-only (needs pynq + an HLS backend) - skipping.')
        return None
    try:
        from pynq.ps import Clocks
    except Exception as exc:
        print(f'could not import pynq Clocks: {exc} - skipping.')
        return None

    image = sample_gray(n)
    kernel = get_kernel(kernel_name)
    ref = conv_reference(image, kernel)
    accel = get_backend('hls_naive', **HW_KW)

    records = []
    try:
        for target in clocks_mhz:
            Clocks.fclk0_mhz = target
            actual = round(float(Clocks.fclk0_mhz), 1)
            out, ms = accel.run(image, kernel, color=False)
            wrong = int(np.count_nonzero(out != ref))
            records.append({'mhz': actual, 'ms': ms, 'ok': wrong == 0, 'wrong': wrong})
            print(f'  fclk0 ~ {actual:6.1f} MHz   {ms:8.3f} ms   '
                  f'{"OK" if wrong == 0 else f"{wrong} wrong px"}')
    finally:
        Clocks.fclk0_mhz = 100
        print('restored fclk0 to 100 MHz')

    mhz = [r['mhz'] for r in records]; ms = [r['ms'] for r in records]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(mhz, ms, '-o', color='steelblue')
    for r in records:
        ax.plot(r['mhz'], r['ms'], 'o', color='green' if r['ok'] else 'red', zorder=3)
    ax.set_xlabel('PL clock fclk0 (MHz)'); ax.set_ylabel('latency (ms)')
    ax.set_title('latency vs clock (green = bit-exact, red = wrong)')
    ax.grid(alpha=0.3); plt.show()
    return records


_ = clock_ramp()

Latency falls as ~1/f: the clock is the speed. The correctness colour is the fuzzy part -
don't expect a textbook cliff on warm silicon. The timing/critical-path story is in
`docs/observability.md` and Act 3's slides.